In [1]:
import warnings
import os 
from pathlib import Path
import re
from natsort import natsort_keygen

import pandas as pd
import anndata as ad 
import numpy as np 
from dotenv import dotenv_values


warnings.simplefilter(action='ignore', category=FutureWarning)

In [3]:
# Setting up filepaths from config
CONFIG = dotenv_values("config.env") 
DATADIR = Path(CONFIG['DATADIR'])

OUTDIR = DATADIR / "publication" / "Supplemental" / "tables"
os.makedirs(OUTDIR, exist_ok=True)

SC_DIR = DATADIR / "9_objects" / "singlecell"
SC_IN_FP = SC_DIR / 'hypomap3d_singlecell_all.h5ad'

TRANSLATOR_FP = DATADIR / "10_analyses" / "clustering_comparison" / 'HypoMap3D_to_other_atlas_celltype_mapper.csv'

RELMASS_DIR = DATADIR / "7_spatial" / "analyses"
RELMASS_FP = RELMASS_DIR / "regionwise_rm_all_wo_root.csv" 
RELMASS_HY_FP = RELMASS_DIR / "regionwise_rm_HY_wo_root_or_fibers.csv"

REGION_META_FP = DATADIR / "7_spatial" / "abc_region_metadata.csv"

FLAGS_FP = RELMASS_DIR / "spatial_flags.csv"
DBSCAN_METRICS_FP = RELMASS_DIR / "dbscan_metrics.csv"

DATASET_META_FP = "resources/dataset_metadata.csv" # assuming kernel position at repo root

SAMPLE_META_FP = "resources/sample_metadata.csv"
PB_META_FP = DATADIR / "9_objects" / "pseudobulks" / "hypomap3d_pseudobulk_all_metadata.csv"

GENE_META_FP = DATADIR / "9_objects" / "gene_metadata.csv"

MARKERS_FP = DATADIR / "9_objects" / "markers" / "hypomap3d_cell_type_marker_genes_all_levels.csv"

QC_ADATAS_DIR = DATADIR / "2b_clean"

SEX_DIR = DATADIR / "10_analyses" / "sexual_dimorphism"
SEX_DEG_FP = SEX_DIR / "L4_multivariate_sex_edgeR_results_significant_only.csv"
CT_SEX_SUMMARY_FP = SEX_DIR / "L4_sd_metrics.csv"
SEX_GO_FP =  SEX_DIR / "GO_results" / "compiled_table.csv"

DMH_OW_FP = DATADIR / "publication" / "Fig_03" / "dmh_cluster_overview.tsv"

In [6]:
adata = ad.read_h5ad(SC_IN_FP, backed="r")

In [4]:
translator = pd.read_csv(TRANSLATOR_FP)

In [5]:
translator.head(2)

,source_label,source_clustering,target_clustering,most_probable_match,most_probable_match_counts_absolute,most_probable_match_counts_relative,second_most_probable_match,second_most_probable_match_counts_absolute,second_most_probable_match_counts_relative,total_num_cells,rest_probabilities
0,Gb1-2 GABA,L1_descr,class_id_label,12 HY GABA,62872,0.812604,11 CNU-HYa GABA,14499,0.187396,77371,0.000000
1,Gl1-1 Glut,L1_descr,class_id_label,14 HY Glut,56698,0.746517,16 HY MM Glut,12398,0.163239,75950,0.090244


# Cluster overview

### Init long-format labelmat

In [9]:
abc_cols = [f"L{i}_abc_like" for i in range(1,6)]
descr_cols = [f"L{i}_descr" for i in range(1,6)]

abc_labelmat = pd.melt(adata.obs.loc[:, abc_cols].drop_duplicates().reset_index(drop=True), value_name="cluster_abc_like").drop_duplicates().drop(columns="variable").reset_index(drop=True)
descr_labelmat = pd.melt(adata.obs.loc[:, descr_cols].drop_duplicates().reset_index(drop=True), value_name="cluster_short").drop_duplicates().drop(columns="variable").reset_index(drop=True)

abc_labelmat["ID"] = abc_labelmat["cluster_abc_like"].apply(lambda x: x.split(" ")[0])
descr_labelmat["ID"] = descr_labelmat["cluster_short"].apply(lambda x: x.split(" ")[0])

abc_labelmat = abc_labelmat.set_index("ID")
descr_labelmat = descr_labelmat.set_index("ID")

labelmat = pd.concat([abc_labelmat, descr_labelmat], axis=1).reset_index()
labelmat["resolution"] = labelmat.ID.apply(lambda x: f"L{x[2]}")
labelmat.head()

,ID,cluster_abc_like,cluster_short,resolution
0,Gl1-1,Gl1-1 Glut,Gl1-1 Glut,L1
1,Gb1-2,Gb1-2 GABA,Gb1-2 GABA,L1
2,NN1-2,NN1-2 Immune,NN1-2 Immune,L1
3,NN1-3,NN1-3 Vascular,NN1-3 Vascular,L1
4,NN1-1,NN1-1 OPC-Oligo,NN1-1 OPC-Oligo,L1


In [9]:
labelmat[labelmat.cluster_short.str.contains("ARH Agrp")]

,ID,cluster_abc_like,cluster_short,resolution
111,Gb3-228,Gb3-228 TU-ARH Otp Six6 Npy2r GABA_1a,Gb3-228 ARH Agrp,L3
391,Gb4-631,Gb4-631 TU-ARH Otp Six6 Npy2r GABA_1a-2,Gb4-631 ARH Agrp.Ppfibp2,L4
423,Gb4-630,Gb4-630 TU-ARH Otp Six6 Npy2r GABA_1a-1,Gb4-630 ARH Agrp.Tll1,L4
1107,Gb5-829,Gb5-829 TU-ARH Otp Six6 Npy2r GABA_1a-2-3,Gb5-829 ARH Agrp.Ppfibp2.Sst,L5
1141,Gb5-826,Gb5-826 TU-ARH Otp Six6 Npy2r GABA_1a-1-1,Gb5-826 ARH Agrp.Tll1,L5
1221,Gb5-828,Gb5-828 TU-ARH Otp Six6 Npy2r GABA_1a-2-2,Gb5-828 ARH Agrp.Ppfibp2.Lrrc17,L5
1419,Gb5-827,Gb5-827 TU-ARH Otp Six6 Npy2r GABA_1a-2-1,Gb5-827 ARH Agrp.Ppfibp2.Alkal2,L5


### Cell numbers per cluster

In [8]:
ctrl_dfs = []
all_dfs = []

for i in range(1,6):
    all_dfs.append(adata.obs[[f"L{i}_descr", "Author"]].groupby([f"L{i}_descr"])["Author"].size())
    ctrl_dfs.append(adata[adata.obs.IsControl].obs[[f"L{i}_descr", "Author"]].groupby([f"L{i}_descr"])["Author"].size())

out_ctrl = pd.concat(ctrl_dfs, axis=0)
out_ctrl = out_ctrl[~out_ctrl.index.duplicated(keep="first")]

out_all = pd.concat(all_dfs, axis=0)
out_all = out_all[~out_all.index.duplicated(keep="first")]

In [25]:
labelmat["num_cells_all"] = labelmat["cluster_short"].replace(out_all.to_dict())
labelmat["num_cells_ctrl"] = labelmat["cluster_short"].replace(out_ctrl.to_dict())

In [27]:
labelmat.head()

,ID,cluster_abc_like,cluster_short,resolution,num_cells_all,num_cells_ctrl
0,Gl1-1,Gl1-1 Glut,Gl1-1 Glut,L1,480322,279075
1,Gb1-2,Gb1-2 GABA,Gb1-2 GABA,L1,562974,328018
2,NN1-2,NN1-2 Immune,NN1-2 Immune,L1,38453,18115
3,NN1-3,NN1-3 Vascular,NN1-3 Vascular,L1,43245,22077
4,NN1-1,NN1-1 OPC-Oligo,NN1-1 OPC-Oligo,L1,231581,140839


### Best ABC and HMv1 matches

Building mappers

In [ ]:
hmv1_mapper_mask = ((translator.source_clustering == "L2_descr") & (translator.target_clustering == "C66_named_propagated")) | (
    (translator.source_clustering == "L3_descr") & (translator.target_clustering == "C286_named_propagated")) | (
    (translator.source_clustering == "L4_descr") & (translator.target_clustering == "C465_named_propagated")) | (
    (translator.source_clustering == "L5_descr") & (translator.target_clustering == "C465_named_propagated"))

hmv1_mapper = translator.loc[hmv1_mapper_mask, ["source_label", "most_probable_match"]].set_index("source_label").to_dict()["most_probable_match"]

abc_mapper_mask = ((translator.source_clustering == "L1_descr") & (translator.target_clustering == "class_id_label_propagated")) | (
    (translator.source_clustering == "L2_descr") & (translator.target_clustering == "subclass_id_label_propagated")) | (
    (translator.source_clustering == "L3_descr") & (translator.target_clustering == "supertype_id_label_propagated")) | (
    (translator.source_clustering == "L4_descr") & (translator.target_clustering == "cluster_id_label_propagated")) | (
    (translator.source_clustering == "L5_descr") & (translator.target_clustering == "cluster_id_label_propagated"))

abc_mapper = translator.loc[abc_mapper_mask, ["source_label", "most_probable_match"]].set_index("source_label").to_dict()["most_probable_match"]

In [36]:
labelmat["HMv1_best_match"] = labelmat.cluster_short.map(hmv1_mapper)
labelmat["ABC_best_match"] = labelmat.cluster_short.map(abc_mapper)

In [40]:
labelmat.head()

,ID,cluster_abc_like,cluster_short,resolution,num_cells_all,num_cells_ctrl,HMv1_best_match,ABC_best_match
0,Gl1-1,Gl1-1 Glut,Gl1-1 Glut,L1,480322,279075,NaN,14 HY Glut
1,Gb1-2,Gb1-2 GABA,Gb1-2 GABA,L1,562974,328018,NaN,12 HY GABA
2,NN1-2,NN1-2 Immune,NN1-2 Immune,L1,38453,18115,NaN,34 Immune
3,NN1-3,NN1-3 Vascular,NN1-3 Vascular,L1,43245,22077,NaN,33 Vascular
4,NN1-1,NN1-1 OPC-Oligo,NN1-1 OPC-Oligo,L1,231581,140839,NaN,31 OPC-Oligo


### Top regions

In [75]:
rm_all = pd.read_csv(RELMASS_FP, index_col=0) 
rm_hy = pd.read_csv(RELMASS_HY_FP, index_col=0) 

# drop HY 
rm_all = rm_all.loc[:, [c for c in rm_all.columns if c != "HY"]]
rm_hy = rm_hy.loc[:, [c for c in rm_hy.columns if c != "HY"]]

In [76]:
rm_all_mapper = {
    cluster: rm_all.loc[cluster].nlargest(5).index.tolist()
    for cluster in rm_all.index
}

rm_hy_mapper = {
    cluster: rm_hy.loc[cluster].nlargest(5).index.tolist()
    for cluster in rm_hy.index
}

In [77]:
labelmat["top_regions_all_nuclei"] = labelmat["ID"].map(rm_all_mapper)
labelmat["top_regions_HY_nuclei"] = labelmat["ID"].map(rm_hy_mapper)

labelmat["top_regions_all_nuclei"] = labelmat["top_regions_all_nuclei"].apply(lambda x: "; ".join(x))
labelmat["top_regions_HY_nuclei"] = labelmat["top_regions_HY_nuclei"].apply(lambda x: "; ".join(x))

In [78]:
col_order = ["resolution", "ID", "cluster_abc_like", "cluster_short", "num_cells_ctrl", "num_cells_all", 
    "ABC_best_match", "HMv1_best_match", "top_regions_all_nuclei", "top_regions_HY_nuclei",]

labelmat = labelmat.loc[:,col_order]

In [79]:
labelmat.head()

,resolution,ID,cluster_abc_like,cluster_short,num_cells_ctrl,num_cells_all,ABC_best_match,HMv1_best_match,top_regions_all_nuclei,top_regions_HY_nuclei
0,L1,Gl1-1,Gl1-1 Glut,Gl1-1 Glut,279075,480322,14 HY Glut,NaN,MEA; LHA; MB; VPM; VMH,LHA; VMH; PH; ZI; AHN
1,L1,Gb1-2,Gb1-2 GABA,Gb1-2 GABA,328018,562974,12 HY GABA,NaN,CP; OT; ACB; SI; BST,ZI; LHA; AHN; MPO; MPN
2,L1,NN1-2,NN1-2 Immune,NN1-2 Immune,18115,38453,34 Immune,NaN,SI; CP; ZI; LHA; GPe,ZI; LHA; AHN; LPO; MPO
3,L1,NN1-3,NN1-3 Vascular,NN1-3 Vascular,22077,43245,33 Vascular,NaN,CP; SI; MB; MEA; LHA,LHA; ZI; PH; TU; LPO
4,L1,NN1-1,NN1-1 OPC-Oligo,NN1-1 OPC-Oligo,140839,231581,31 OPC-Oligo,NaN,int; CP; LHA; MB; ZI,LHA; ZI; PH; AHN; LPO


In [80]:
labelmat.to_csv(OUTDIR / "supp_table_5_cluster_overview.csv", index=False)

# Acronyms

In [81]:
region_meta = pd.read_csv(REGION_META_FP)
region_meta = region_meta.loc[:, ["acronym", "name"]]

In [83]:
region_meta.head()

,acronym,name
0,HY,Hypothalamus
1,ME,Median eminence
2,LZ,Hypothalamic lateral zone
3,ZI,Zona incerta
4,FF,Fields of Forel


In [84]:
region_meta.to_csv(OUTDIR / "supp_table_7_ABC_acronyms.csv", index=False)

# Spatial characteristics

In [ ]:
flags = pd.read_csv(FLAGS_FP)
flags.warning = flags.warning.fillna("None")
flags_mapper = flags.set_index("cluster_id").to_dict()["warning"]

dbscan = pd.read_csv(DBSCAN_METRICS_FP)
dbscan["cluster_id"] = dbscan.cluster_name.apply(lambda x: x.split(" ")[0])
otsu_mapper = dbscan.loc[:, ["cluster_id", "otsu_thr"]].drop_duplicates().set_index("cluster_id").to_dict()["otsu_thr"]

In [100]:
# Remove DBSCAN outliers
df_valid = dbscan[dbscan["dbscan_label"] != -1].copy()

# Count unique (non-outlier) DBSCAN clusters per cluster_id
n_dbscan_clusters = (
    df_valid.groupby("cluster_id")["dbscan_label"]
    .nunique()
    .rename("n_dbscan_clusters")
)

# Rank clusters by size within each cluster_id
df_valid = df_valid.sort_values(
    ["cluster_id", "num_vox"],
    ascending=[True, False]
)

df_valid["rank"] = (
    df_valid.groupby("cluster_id")
    .cumcount()
)

# Keep only the two largest clusters
top2 = df_valid[df_valid["rank"] < 2].copy()

# Map rank to readable suffixes
rank_names = {0: "biggest", 1: "second_biggest"}
top2["rank"] = top2["rank"].map(rank_names)

# Pivot to wide format
wide = (
    top2.set_index(["cluster_id", "rank"])[
        ["num_vox", "center_coords", "rel_mass"]
    ]
    .unstack("rank")
)

# Flatten MultiIndex columns
wide.columns = [
    f"{col}_{rank}"
    for col, rank in wide.columns
]

wide = wide.reset_index()

# Add total number of DBSCAN clusters
result = wide.merge(
    n_dbscan_clusters,
    on="cluster_id",
    how="left"
)

In [111]:
result["flag"] = result.cluster_id.map(flags_mapper)
result["otsu_thr"] = result.cluster_id.map(otsu_mapper)

In [123]:
col_order = ["cluster_id", "flag", "otsu_thr", "n_dbscan_clusters",
"num_vox_biggest", "rel_mass_biggest", "center_coords_biggest",
"num_vox_second_biggest", "rel_mass_second_biggest", "center_coords_second_biggest"]

result = result.loc[:, col_order]

In [126]:
natkey = natsort_keygen()

celltype_order = {"Gb": 0, "Gl": 1, "NN": 2}

def cluster_sort_key(cid):
    m = re.match(r"([A-Za-z]{2})(\d+)-(.*)", cid)
    if m is None:
        return (float("inf"), float("inf"), "")
    
    celltype, level, rest = m.groups()
    return (
        int(level),                              # sort by level
        celltype_order.get(celltype, 99),        # Gb -> Gl -> NN
        natkey(rest)                             # natural sort after the hyphen
    )

result = (
    result.sort_values(
        by="cluster_id",
        key=lambda s: s.map(cluster_sort_key)
    )
    .reset_index(drop=True)
)

In [130]:
result.head(2)

,cluster_id,flag,otsu_thr,n_dbscan_clusters,num_vox_biggest,rel_mass_biggest,center_coords_biggest,num_vox_second_biggest,rel_mass_second_biggest,center_coords_second_biggest
0,Gb1-2,None,0.36083,22,655476.0,0.618039,[5.14848328 5.52823097 4.14191378],13981.0,0.009307,[7.42651813 4.57991918 3.50426293]
1,Gl1-1,None,0.21526,31,809906.0,0.708729,[6.99053007 5.42888591 4.61402339],2793.0,0.002442,[4.54222163 5.92836556 3.12360365]


In [131]:
result.to_csv(OUTDIR / "supp_table_X_spatial_characteristics.csv", index=False)

# Regionwise relative masses

In [132]:
rm_all_fresh = pd.read_csv(RELMASS_FP, index_col=0) 

In [220]:
rm_all_fresh.head(2)

,TMv,int,IF,PT,AAA,IGL,PVa,PVH,mfb,ACB,...,MMp,MMd,PN,IPR,IPC,IPL,IPI,IPDM,IPDL,IPRL
Gb1-2,0.001161,0.011962,0.000149,0.000669,0.008502,0.000025,0.001321,0.001105,0.000140,0.060894,...,0.000014,0.000133,0.000041,0.000380,0.000033,0.000051,0.000042,4.281577e-07,0.000042,0.000097
Gl1-1,0.001100,0.010536,0.001118,0.006645,0.005775,0.000036,0.000748,0.007419,0.001143,0.003039,...,0.002030,0.002900,0.000129,0.001204,0.000074,0.000118,0.000074,6.246938e-06,0.000171,0.000400


In [134]:
rm_all_fresh.to_csv(OUTDIR / "supp_table_6_region_rel_mass.csv")

# Dataset metadata 

In [216]:
ds_meta = pd.read_csv(DATASET_META_FP)

In [217]:
ds_meta.head()

,ReferenceID,Author,Source,Data_source_link,GEO,Publication_Link,Suspension,Technology
0,SC1-01,Affinati,SRA,NaN,GSE172204,https://elifesciences.org/articles/69065,snRNAseq,10x v3
1,SC1-02,Anderson,SRA,NaN,GSE193921,https://doi.org/10.1016/j.neuron.2021.12.002,scRNAseq,10x v3
2,SC1-03,Campbell,SRA,NaN,GSE93374,https://www.ncbi.nlm.nih.gov/pubmed/28166221,scRNAseq,Drop-seq
3,SC1-04,Chen,SRA,NaN,GSE87544,https://www.ncbi.nlm.nih.gov/pubmed/28355573,scRNAseq,Drop-seq
4,SC1-05,Dowsett,SRA,NaN,GSE207736,https://www.nature.com/articles/s42255-022-006...,snRNAseq,10x v3


In [218]:
ds_meta.to_csv(OUTDIR / "supp_table_1_dataset_meta.csv", index=False)

# Sample metadata

The most up-to-date sample metadata is available for the pseudobulks.

However, samples can be dropped during the QC. Hence, we have to supplement the pseudobulk metadata with the input meta data

In [4]:
pb_meta = pd.read_csv(PB_META_FP, )

In [5]:
pb_samples = pb_meta.loc[:, ['Sample', 'Author_Sex_harmonized','sex_per_sample_inferred', 'Strain', 'Diet', 
       'Treatment',  'Age', 'IsControl', "Mean_age_in_weeks"]]

In [6]:
pb_samples = pb_samples.rename(columns = {
    "sex_per_sample_inferred": "Sex_per_sample_inferred", 
    "Treatment_harmonized": "Treatment",
    "Author_Sex_harmoinzed": "Author_sex",
})

Some samples were filtered during QC, so we have to add them back.

In [ ]:
sample_meta = pd.read_csv(SAMPLE_META_FP)

lost = [s for s in sample_meta.Sample.unique() if s not in pb_samples.Sample.unique()]

lost_ones = sample_meta.loc[sample_meta.Sample.isin(lost)]

In [8]:
lost_ones.IsControl = lost_ones.IsControl.replace({"N": False})

/tmp/ipykernel_1056427/444502968.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lost_ones.IsControl = lost_ones.IsControl.replace({"N": False})


In [9]:
meta_out = pd.concat([pb_samples, lost_ones], axis=0)

In [10]:
meta_out = meta_out.sort_values(by="Sample")

In [11]:
meta_out = meta_out.rename(columns = {
    "Sample": "SampleID",
    "Author_Sex_harmonized": "Author_Sex",
    "Age": "Age_harmonized",
    
})

In [12]:
meta_out["BioSampleID"] = meta_out.SampleID.apply(lambda x: "_".join(x.split("_")[1:]))

In [13]:
meta_out = meta_out.loc[:,[
    "SampleID", "BioSampleID", "Author_Sex", "Strain", "Diet", "Treatment", "IsControl", "Age_harmonized", "Mean_age_in_weeks"
]]

In [14]:
meta_out.tail()

,SampleID,BioSampleID,Author_Sex,Strain,Diet,Treatment,IsControl,Age_harmonized,Mean_age_in_weeks
364,SC2-25_SAMN46505605,SAMN46505605,male,C57BL/6J,HFD_12weeks,Ad lib. HFD + acyl-GLP-1,False,36 weeks,36.0
365,SC2-25_SAMN46505606,SAMN46505606,male,C57BL/6J,HFD_12weeks,Ad lib. HFD + GIPR antagonist,False,36 weeks,36.0
366,SC2-25_SAMN46505607,SAMN46505607,male,C57BL/6J,HFD_12weeks,Ad lib. HFD + acyl-GIP,False,36 weeks,36.0
367,SC2-25_SAMN46505608,SAMN46505608,male,C57BL/6J,HFD_12weeks,Ad lib. HFD + vehicle,False,36 weeks,36.0
368,SC2-25_SAMN46505609,SAMN46505609,male,C57BL/6J,Normal chow,Ad lib. NC + vehicle,True,36 weeks,36.0


In [15]:
meta_out.to_csv(OUTDIR /"supp_table_4_sample_metadata.csv", index=False)

# Gene metadata

In [12]:
gene_meta = pd.read_csv(GENE_META_FP)

In [13]:
gene_meta = gene_meta.rename(columns = {
    "gene_name": "Gene name", 
    "gene_ids": "Gene stable ID",
    "IsHvg": "IsHVG",
    "IsHvgExcludedGene": "IsExcludedGene"
})

In [14]:
gene_meta = gene_meta.loc[:, ['Gene name', 'Gene stable ID', 'IsHVG', 'IsMouseTF', 'IsHumanTFOrtholog',
       'IsSourceGene', 'IsTargetGene', 'IsTopGene', 'IsExcludedGene',]]

In [15]:
gene_meta.head(2)

,Gene name,Gene stable ID,IsHVG,IsMouseTF,IsHumanTFOrtholog,IsSourceGene,IsTargetGene,IsTopGene,IsExcludedGene
0,Gm38212,ENSMUSG00000104478,False,False,False,False,False,False,False
1,Gm7449,ENSMUSG00000104385,False,False,False,False,False,False,False


In [16]:
gene_meta.to_csv(OUTDIR / "supp_table_9_gene_meta.csv", index=False)

# Marker genes

In [5]:
markers = pd.read_csv(MARKERS_FP)

In [6]:
markers = markers.rename(columns={ # rename for consistency with supplementals of first submission
    "celltype_descr": "cluster_short",
    "celltype_abc_like": "cluster_abc_like",
    "log2FC": "avg_log2FC",
    "VE": "VE_stat",
    "p": "pvalue"
})

In [7]:
keep_cols = ["cluster_short", "cluster_abc_like", "gene_name", "specificity", "avg_log2FC", 
    "VE_stat", "padj", "pvalue", "pctA", "pctB", "comparison"]

### One Vs All: top 30

Getting the top 20 for the case where the xlsx becomes too long.

In [8]:
ova_markers = markers.loc[(markers.comparison=="OneVsAll") & (markers.padj <= 0.05), keep_cols]

In [9]:
ova_markers_top30 = (
    ova_markers
    .groupby("cluster_short", group_keys=False)
    .apply(lambda g: g.nlargest(30, "specificity"))
)

In [10]:
ova_markers_top20 = (
    ova_markers
    .groupby("cluster_short", group_keys=False)
    .apply(lambda g: g.nlargest(20, "specificity"))
)

In [11]:
ova_markers_top30.head()

,cluster_short,cluster_abc_like,gene_name,specificity,avg_log2FC,VE_stat,padj,pvalue,pctA,pctB,comparison
371493,Gb1-2 GABA,Gb1-2 GABA,Dlx6os1,4.985779,5.394670,82.441806,0.0,0.0,0.329668,0.012986,OneVsAll
371494,Gb1-2 GABA,Gb1-2 GABA,Dlx2,4.931347,5.232152,37.522986,0.0,0.0,0.157875,0.004673,OneVsAll
371495,Gb1-2 GABA,Gb1-2 GABA,Arx,4.607842,4.984361,30.011273,0.0,0.0,0.133279,0.005232,OneVsAll
371496,Gb1-2 GABA,Gb1-2 GABA,Dlx1,4.582769,5.055209,66.847075,0.0,0.0,0.277561,0.013606,OneVsAll
371497,Gb1-2 GABA,Gb1-2 GABA,Dlx6,4.468016,4.840443,43.495062,0.0,0.0,0.188435,0.007539,OneVsAll


In [12]:
len(ova_markers_top20)

39280

In [13]:
len(ova_markers_top30)

58920

### One vs Sisters: top 10

In [14]:
ovs_markers = markers.loc[(markers.comparison=="OneVsSisters") & (markers.padj <= 0.05), keep_cols]

ovs_markers_top10 = (
    ovs_markers
    .groupby("cluster_short", group_keys=False)
    .apply(lambda g: g.nlargest(10, "specificity"))
)

In [15]:
len(ovs_markers_top10)

19629

### Region: top 10 for each region and cluster

In [16]:
r_markers = markers.loc[
    (~markers["comparison"].isin(["OneVsSisters", "OneVsAll"]))
    & (markers["padj"] <= 0.05),
    keep_cols,
]

r_markers_top10 = (
    r_markers
    .sort_values("specificity", ascending=False)
    .groupby(["cluster_short", "comparison"], group_keys=False)
    .head(10)
)

In [17]:
r_markers_top10.comparison = r_markers_top10.comparison.apply(lambda x: f"OneVs{x}_region")

In [18]:
len(r_markers_top10)

27486

### Concat and save

In [19]:
markers_out_30 = pd.concat([ova_markers_top30, ovs_markers_top10, r_markers_top10], axis=0)
markers_out_20 = pd.concat([ova_markers_top20, ovs_markers_top10, r_markers_top10], axis=0)

In [22]:
markers_out_30["pct_diff"] = np.abs(markers_out_30.pctA - markers_out_30.pctB) / (
                                markers_out_30.pctA + markers_out_30.pctB + 1e-10)

markers_out_20["pct_diff"] = np.abs(markers_out_20.pctA - markers_out_20.pctB) / (
                                markers_out_20.pctA + markers_out_20.pctB + 1e-10)

In [24]:
markers_out_20.columns

Index(['cluster_short', 'cluster_abc_like', 'gene_name', 'specificity',
       'avg_log2FC', 'VE_stat', 'padj', 'pvalue', 'pctA', 'pctB', 'comparison',
       'pct_diff'],
      dtype='object')

In [28]:
markers_order = ['cluster_abc_like', 'cluster_short', 'gene_name', 'specificity',
       'avg_log2FC', 'VE_stat', 'padj', 'pvalue', 'pctA', 'pctB', 'pct_diff', 'comparison',
       ]
markers_out_20 = markers_out_20.loc[:, markers_order]
markers_out_30 = markers_out_30.loc[:, markers_order]

In [29]:
markers_out_20.head(2)

,cluster_abc_like,cluster_short,gene_name,specificity,avg_log2FC,VE_stat,padj,pvalue,pctA,pctB,pct_diff,comparison
371493,Gb1-2 GABA,Gb1-2 GABA,Dlx6os1,4.985779,5.394670,82.441806,0.0,0.0,0.329668,0.012986,0.924205,OneVsAll
371494,Gb1-2 GABA,Gb1-2 GABA,Dlx2,4.931347,5.232152,37.522986,0.0,0.0,0.157875,0.004673,0.942508,OneVsAll


In [30]:
markers_out_20.to_csv(OUTDIR / "supp_table_8_marker_genes_20.csv", index=False)
markers_out_30.to_csv(OUTDIR / "supp_table_8_marker_genes_30.csv", index=False)

# QC thresholds

Let's compile the thresholds used for filtering. They can be retrieved from the filtered adatas' .uns slots

In [276]:
rows = []

for h5ad_file in sorted(QC_ADATAS_DIR.glob("SC*.h5ad")):
    adata = ad.read_h5ad(h5ad_file, backed="r")

    rows.append({
        "ReferenceID": h5ad_file.stem,
        "Max % counts mitochondrial": adata.uns["max_pct_mito_threshold"],
        "Min total counts per cell": adata.uns["min_counts_threshold"],
        "Min genes per cell": adata.uns["min_genes_threshold"],
    })

qc_thresholds = pd.DataFrame(rows)

In [278]:
qc_thresholds.head(3)

,ReferenceID,Max % counts mitochondrial,Min total counts per cell,Min genes per cell
0,SC1-01,5,1500,300
1,SC1-02,5,2750,300
2,SC1-03,5,1000,300


In [ ]:
qc_thresholds.to_csv(OUTDIR / "supp_table_2_qc_thresholds.csv", index=False)

# Sex analysis

### Sex DEGs

In [32]:
sex_deg = pd.read_csv(SEX_DEG_FP)

In [43]:
sex_deg = sex_deg.rename(columns={
    "Unnamed: 0": "Gene",
    "id": "Cluster"
})

sex_deg = sex_deg.loc[:, ["Cluster", "Gene", "logFC", "logCPM", "F", "PValue", "FDR"]].sort_values(by="Cluster")

In [45]:
sex_deg.head(2)

,Cluster,Gene,logFC,logCPM,F,PValue,FDR
2568,Gb4-319,Eif2s3y,7.356074,6.366862,376.911119,2.880831e-20,3.292790e-17
2567,Gb4-319,Uty,6.472098,6.456550,427.220780,4.220965e-26,7.236845e-23


In [46]:
sex_deg.to_csv(OUTDIR / "supp_table_10_sex_degs.csv", index=False)

### Sex cluster summary

In [51]:
sex_sum = pd.read_csv(CT_SEX_SUMMARY_FP)

sex_sum = sex_sum.rename(columns={
    "celltype_ID": "Cluster",
    "pct_Female": "pct_female",
    "sLISI_Q1" : "sex_sLISI_Q1",
    "sLISI_Q2" : "sex_sLISI_Q2",
    "sLISI_Q3" : "sex_sLISI_Q3",
})

sex_sum = sex_sum.loc[:, [
    "Cluster", "pct_female", "sex_ratio_deviance", "sex_sLISI_Q1",
    "sex_sLISI_Q2", "sex_sLISI_Q3", "n_DEGs"
]]

In [53]:
sex_sum.head(2)

,Cluster,pct_female,sex_ratio_deviance,sex_sLISI_Q1,sex_sLISI_Q2,sex_sLISI_Q3,n_DEGs
0,Gl4-220,0.325898,0.174102,0.310767,0.808747,0.981644,561
1,Gl4-140,0.409053,0.090947,0.545246,0.877870,0.982116,539


In [54]:
sex_sum.to_csv(OUTDIR / "supp_table_12_sex_cluster_summary.csv", index=False)

### Sex GO terms

In [14]:
sex_go = pd.read_csv(SEX_GO_FP)

In [34]:
col_renamer = {
    "native": "term_id",
    "name": "term_name",
    "target": "cluster"
}

In [31]:
common = sex_go.loc[sex_go.target=="global", [
    "query", "significant", "p_value", "term_size", "query_size", "intersection_size",
    "precision", "recall", "native", "source", "name", "effective_domain_size", "source_order", "highlighted" 
]].reset_index(drop=True).reset_index(names="Index")
common.Index += 1
common = common.rename(columns=col_renamer)

In [32]:
common.head(2)

,Index,query,significant,p_value,term_size,query_size,intersection_size,precision,recall,term_id,source,term_name,effective_domain_size,source_order,highlighted
0,1,query_1,True,1.047090e-09,72,164,14,0.085366,0.194444,GO:0007156,GO:BP,homophilic cell-cell adhesion,15562,2586,True
1,2,query_1,True,1.501628e-07,954,164,37,0.225610,0.038784,GO:0007155,GO:BP,cell adhesion,15562,2585,False


In [33]:
common.to_csv(OUTDIR / "supp_table_11_common_sex_go.csv", index=False)

In [39]:
cluster_go = sex_go.loc[sex_go.target!="global", ["target",
    "query", "significant", "p_value", "term_size", "query_size", "intersection_size",
    "precision", "recall", "native", "source", "name", "effective_domain_size", "source_order", "highlighted" 
]].reset_index(drop=True).reset_index(names="Index")
cluster_go.Index += 1
cluster_go = cluster_go.rename(columns=col_renamer)
cluster_go["sex"] = "both"
cluster_go = cluster_go.loc[:, [
    "cluster", "Index", "sex", "query", "significant", "p_value", "term_size", "query_size", "intersection_size",
    "precision", "recall", "term_id", "source", "term_name", "effective_domain_size", "source_order", "highlighted" 
]]

In [41]:
cluster_go.head(2)

,cluster,Index,sex,query,significant,p_value,term_size,query_size,intersection_size,precision,recall,term_id,source,term_name,effective_domain_size,source_order,highlighted
0,Gb4-444,1,both,query_1,True,0.000001,598,258,47,0.182171,0.078595,GO:0007267,GO:BP,cell-cell signaling,9473,2659,True
1,Gb4-444,2,both,query_1,True,0.000027,457,258,38,0.147287,0.083151,GO:0099536,GO:BP,synaptic signaling,9473,17869,False


In [42]:
cluster_go.to_csv(OUTDIR / "supp_table_13_celltype_sex_go.csv", index=False)

# DMH overview

In [4]:
dmh_cluster_overview = pd.read_csv(DMH_OW_FP, delimiter="\t")

In [7]:
dmh_out = dmh_cluster_overview.loc[:, ["L3_ID", "L3_descr", "L2_descr", "region_rank", "relative_mass",]]

In [12]:
descr_to_abc_mapper = labelmat.set_index("cluster_short").to_dict()["cluster_abc_like"]

In [13]:
dmh_out["L3_abc_like"] = dmh_out.L3_descr.map(descr_to_abc_mapper)
dmh_out["L2_abc_like"] = dmh_out.L2_descr.map(descr_to_abc_mapper)

In [15]:
dmh_out = dmh_out.rename(columns={"L3_descr": "L3_short", "L2_descr": "L2_short"})

In [17]:
dmh_out = dmh_out.loc[:, ["L3_ID", "L3_short", "L3_abc_like", "L2_short", "L2_abc_like", "relative_mass", "region_rank"]]

In [19]:
dmh_out.head(2)

,L3_ID,L3_short,L3_abc_like,L2_short,L2_abc_like,relative_mass,region_rank
0,Gl3-84,Gl3-84 DMH Nr2f1,Gl3-84 DMH Nkx2-4 Glut_2,Gl2-26 DMH Nkx2-4,Gl2-26 DMH Nkx2-4 Glut,0.256136,1
1,Gl3-83,Gl3-83 DMH Pdyn,Gl3-83 DMH Nkx2-4 Glut_3,Gl2-26 DMH Nkx2-4,Gl2-26 DMH Nkx2-4 Glut,0.287700,1


In [20]:
dmh_out.to_csv(OUTDIR / "supp_table_X_dmh_clusters.csv", index=False)